# Extract Emotion Vectors


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"  # must be before import torch

import json
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import sys
sys.path.append("..")  # adjust if config.py is in a different relative path

from config import DEFAULT_MODEL, TARGET_LAYER, TOKEN_START, BATCH_SIZE, STORIES_JSONL, VECTORS_OUT, RAW_MEANS_OUT

print(f"Model      : {DEFAULT_MODEL}")
print(f"Layer      : {TARGET_LAYER}")
print(f"Token start: {TOKEN_START}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Input      : {STORIES_JSONL}")
print(f"Output     : {VECTORS_OUT}")

## Load Model


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

## Load Stories


In [ ]:
def load_stories(path):
    data = defaultdict(list)
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            data[rec["emotion"]].append(rec["text"])
    return dict(data)

stories_by_emotion = load_stories(Path(STORIES_JSONL))
emotions = sorted(stories_by_emotion.keys())

print(f"{len(emotions)} emotions, {sum(len(v) for v in stories_by_emotion.values())} total stories")
for e in emotions:
    print(f"  {e:<20} {len(stories_by_emotion[e]):>4} stories")


## Extract Layer-21 Activations


In [ ]:
def mean_pool_from(hidden, start):
    sliced = hidden[start:]
    if sliced.shape[0] == 0:
        sliced = hidden
    return sliced.mean(dim=0)

@torch.inference_mode()
def get_layer_mean(texts, layer, token_start, batch_size):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        layer_h = out.hidden_states[layer].float().cpu()  # (B, T, D)
        mask    = enc["attention_mask"].cpu()
        for b in range(layer_h.shape[0]):
            seq_len = mask[b].sum().item()
            h   = layer_h[b, :seq_len, :]
            vec = mean_pool_from(h, token_start)
            all_vecs.append(vec)
    return torch.stack(all_vecs, dim=0)  # (N, D)

emotion_raw_means = {}

for emotion in tqdm(emotions, desc="Emotions"):
    texts = stories_by_emotion[emotion]
    vecs  = get_layer_mean(texts, layer=TARGET_LAYER, token_start=TOKEN_START, batch_size=BATCH_SIZE)
    emotion_raw_means[emotion] = vecs.mean(dim=0)  # (D,)
    tqdm.write(f"  {emotion}: {vecs.shape} -> mean shape {emotion_raw_means[emotion].shape}")

print(f"\nDone. Vector dim: {next(iter(emotion_raw_means.values())).shape[0]}")


## Center: Subtract Cross-Emotion Mean


In [ ]:
all_means          = torch.stack([emotion_raw_means[e] for e in emotions], dim=0)  # (E, D)
cross_emotion_mean = all_means.mean(dim=0)                                          # (D,)

emotion_vectors = {
    e: emotion_raw_means[e] - cross_emotion_mean
    for e in emotions
}

# Quick sanity check — norms should all be similar
for e in emotions:
    norm = emotion_vectors[e].norm().item()
    print(f"  {e:<20} norm={norm:.4f}")


## Save


In [ ]:
Path(VECTORS_OUT).parent.mkdir(parents=True, exist_ok=True)

torch.save(emotion_vectors,   VECTORS_OUT)
torch.save(emotion_raw_means, RAW_MEANS_OUT)

print(f"Saved centered vectors  -> {VECTORS_OUT}")
print(f"Saved raw means         -> {RAW_MEANS_OUT}")
print(f"Vector shape: {next(iter(emotion_vectors.values())).shape}")


## Similarity Check


In [ ]:
import torch.nn.functional as F

# Build normalised matrix
mat = torch.stack([emotion_vectors[e] for e in emotions], dim=0).float()
mat = F.normalize(mat, dim=1)
sim = (mat @ mat.T).numpy()  # (E, E)

# Print top-3 most similar emotions for a few examples
for i, e in enumerate(emotions):
    sims = [(emotions[j], sim[i, j]) for j in range(len(emotions)) if j != i]
    top3 = sorted(sims, key=lambda x: -x[1])[:3]
    top3_str = ", ".join(f"{name} ({s:.2f})" for name, s in top3)
    print(f"  {e:<20} closest: {top3_str}")
